# Notebook 5 — Stage 3: Fine-Tuning Qwen2.5-VL with LoRA
**MAI 656 — Natural Language Processing | Canadian University Dubai | Spring 2026**

This notebook fine-tunes `Qwen/Qwen2.5-VL-7B-Instruct` on your Arabic handwriting dataset using **4-bit quantization + LoRA**.

> ⚠️ **A GPU with ≥24 GB VRAM is required.** Run this on the AWS g5.xlarge instance (NVIDIA A10G) or a Colab A100.

**Input:**
- `data/train/train.jsonl`
- `data/eval/eval.jsonl`

**Output:** `models/lora_adapter/` — saved LoRA adapter (~380 MB)

**Expected training time:** ~2–4 hours on g5.xlarge

## 1. Check GPU

In [2]:
!nvidia-smi

Fri Apr 17 21:28:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.16             Driver Version: 580.126.16     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A10G                    On  |   00000000:00:1E.0 Off |                    0 |
|  0%   25C    P8             11W /  300W |       0MiB /  23028MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Install Dependencies

In [4]:
!pip install transformers==4.47.0 peft==0.14.0 accelerate==1.2.1 bitsandbytes==0.45.0 --quiet
!pip install qwen-vl-utils --quiet

## 3. Set Project Root

**Colab:** mounts Google Drive and reads from `MyDrive/nlp_project`.  
**Local:** update `LOCAL_PROJECT_ROOT` below to point to your project folder.

In [5]:
import os
from pathlib import Path

IN_COLAB = 'google.colab' in str(get_ipython())

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/nlp_project')
else:
    # ── Set this to your local project directory ──────────────────────────
    PROJECT_ROOT = Path(r'/home/ubuntu/nlp_project')
    # ──────────────────────────────────────────────────────────────────────

assert PROJECT_ROOT.exists(), (
    f'Project root not found: {PROJECT_ROOT}\n'
    'Update LOCAL_PROJECT_ROOT above to match your system.'
)

print(f'{"Colab" if IN_COLAB else "Local"} | Project root: {PROJECT_ROOT}')

Local | Project root: /home/ubuntu/nlp_project


## 4. Configuration

> ⚠️ `MIN_PIXELS`, `MAX_PIXELS`, and `DPI` **must match** what was used in the data transformation step (Notebook 4).

In [ ]:
from pathlib import Path

# --- Model ---
BASE_MODEL     = 'Qwen/Qwen2.5-VL-7B-Instruct'
MAX_SEQ_LENGTH = 8192  # CRITICAL: Do NOT set below 8192!

# --- LoRA ---
LORA_R       = 32    # Rank (max for 24 GB GPU with this model)
LORA_ALPHA   = 64    # Scaling factor (2x rank)
LORA_DROPOUT = 0.05  # Regularization
LORA_TARGET_MODULES = [
    'q_proj', 'k_proj', 'v_proj', 'o_proj',       # Attention
    'gate_proj', 'up_proj', 'down_proj',            # Feed-forward
]

# --- Training ---
NUM_EPOCHS             = 3
BATCH_SIZE             = 1
GRADIENT_ACCUMULATION  = 2    # Effective batch size = 2
LEARNING_RATE          = 2e-5
LR_SCHEDULER           = 'cosine'
WARMUP_RATIO           = 0.1

# --- Image processing (MUST match inference!) ---
MIN_PIXELS = 4   * 28 * 28   # 3,136
MAX_PIXELS = 128 * 28 * 28   # 100,352
DPI = 100

# --- Quantization ---
LOAD_IN_4BIT  = True
QUANT_TYPE    = 'nf4'
DOUBLE_QUANT  = True

# --- Paths ---
TRAIN_DATA = Path(PROJECT_ROOT) / 'data' / 'train' / 'train.jsonl'
EVAL_DATA  = Path(PROJECT_ROOT) / 'data' / 'eval'  / 'eval.jsonl'
OUTPUT_DIR = Path(PROJECT_ROOT) / 'models' / 'lora_adapter'
LOG_DIR    = Path(PROJECT_ROOT) / 'logs'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

print('Configuration loaded.')
print(f'Train data: {TRAIN_DATA}')
print(f'Eval data:  {EVAL_DATA}')
print(f'Output dir: {OUTPUT_DIR}')

## 5. Load Model with 4-bit Quantization

In [ ]:
import torch
from transformers import (
    Qwen2_5_VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig,
)

print('Loading model with 4-bit quantization...')

quantization_config = BitsAndBytesConfig(
    load_in_4bit=LOAD_IN_4BIT,
    bnb_4bit_quant_type=QUANT_TYPE,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=DOUBLE_QUANT,
)

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    BASE_MODEL,
    quantization_config=quantization_config,
    torch_dtype=torch.bfloat16,
    device_map='auto',
    attn_implementation='sdpa',
)

processor = AutoProcessor.from_pretrained(
    BASE_MODEL,
    min_pixels=MIN_PIXELS,
    max_pixels=MAX_PIXELS,
)

print('Model and processor loaded.')

## 6. Apply LoRA

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

print('Applying LoRA...')

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    task_type='CAUSAL_LM',
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 7. Build the Dataset

In [ ]:
import json
from torch.utils.data import Dataset

class ArabicHTRDataset(Dataset):
    def __init__(self, data_path, processor, max_length):
        self.processor = processor
        self.max_length = max_length
        self.samples = []
        with open(data_path) as f:
            for line in f:
                self.samples.append(json.loads(line))
        print(f'Loaded {len(self.samples)} samples from {data_path}')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        messages = sample['messages']

        text = self.processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )

        inputs = self.processor(
            text=text, return_tensors='pt',
            max_length=self.max_length,
            truncation=True, padding='max_length',
        )
        inputs['labels'] = inputs['input_ids'].clone()
        return {k: v.squeeze(0) for k, v in inputs.items()}


print('Loading datasets...')
train_dataset = ArabicHTRDataset(TRAIN_DATA, processor, MAX_SEQ_LENGTH)
eval_dataset  = ArabicHTRDataset(EVAL_DATA,  processor, MAX_SEQ_LENGTH)

print(f'Training samples: {len(train_dataset)}')
print(f'Eval samples:     {len(eval_dataset)}')

if len(train_dataset) == 0:
    raise ValueError('NO TRAINING SAMPLES! MAX_SEQ_LENGTH is probably too low.')

## 8. Train

In [ ]:
import os
from transformers import TrainingArguments, Trainer

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
torch.cuda.empty_cache()

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type=LR_SCHEDULER,
    warmup_ratio=WARMUP_RATIO,
    bf16=True,
    gradient_checkpointing=True,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    logging_steps=10,
    logging_dir=str(LOG_DIR),
    report_to='none',
    dataloader_pin_memory=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
)

print('Starting training...')
trainer.train()

## 9. Save the LoRA Adapter

In [ ]:
print(f'Saving LoRA adapter to {OUTPUT_DIR}...')
model.save_pretrained(str(OUTPUT_DIR))
processor.save_pretrained(str(OUTPUT_DIR))
print('Training complete!')

# Show adapter size
import subprocess
result = subprocess.run(['du', '-sh', str(OUTPUT_DIR)], capture_output=True, text=True)
print(f'Adapter size: {result.stdout.strip()}')

## Troubleshooting

| Problem | Cause | Fix |
|---------|-------|-----|
| CUDA out of memory | LoRA rank too high or images too large | Reduce `LORA_R` to 16, reduce `MAX_PIXELS` |
| Training completed with 0 steps | `MAX_SEQ_LENGTH` too low | Set to 8192 or higher |
| `Killed` / OOM | Batch size too large | Keep `BATCH_SIZE=1`, increase `GRADIENT_ACCUMULATION` |
| Blurry images | DPI too low | Use `DPI=100` in transform step |

## Next Step

LoRA adapter saved to `models/lora_adapter/`. Continue to:

**Notebook 6 → `06_evaluation.ipynb`** — Evaluate the fine-tuned model (CER, WER, exact match)